YOUTUBE CHANNELS DATA COLLECTION USING API

In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = '<YOUR_API_KEY>'
BASE_URL = "https://www.googleapis.com/youtube/v3"

In [3]:
endpoint="search"
search_url=f"{BASE_URL}/{endpoint}"
params_search = {
    "key": API_KEY,
    "part": "snippet",
    "q" : "technology|education|gaming|music|news|finance|sport|cooking|travel|fitness",
    "maxResults" : 50
}

In [4]:
# Gather all channel ids
all_channel_ids = set()
while True:
    response=requests.get(search_url,params=params_search)
    data=response.json()
    channel_ids = list(set(channels['snippet']['channelId'] for channels in data['items']))
    all_channel_ids.update(channel_ids)
    if "nextPageToken" not in data:
        break
    params_search={
        "key": API_KEY,
        "part": "snippet",
        "q" : "technology|education|gaming|music|news|finance|sport|cooking|travel|fitness",
        "maxResults" : 50,
        "pageToken": data['nextPageToken']
    }

In [5]:
len(all_channel_ids)

434

In [6]:
# Gather all channels information using channel ids retrieved above
endpoint="channels"
channel_url=f"{BASE_URL}/{endpoint}"

all_channel_data = []

# Fetch channel details in batches of 50, as the YouTube API accepts up to 50 channel IDs per request
for batch in range(0,len(all_channel_ids),50):
    params_channels = {
        "key": API_KEY,
        "part": "snippet,statistics,status",
        "id":  ','.join(list(all_channel_ids)[batch:batch+50])
    }
    response=requests.get(channel_url,params=params_channels).json()
    if "items" in response:
        all_channel_data.extend(response["items"])

In [11]:
# endpoint="channels"
# channel_url=f"{BASE_URL}/{endpoint}"

# all_channel_data = []

# # Fetch channel details in batches of 50, as the YouTube API accepts up to 50 channel IDs per request
# for batch in range(0,len(all_channel_ids),50):
#     params_channels = {
#         "key": API_KEY,
#         "part": "snippet,statistics,status",
#         "id":  ','.join(list(all_channel_ids)[batch:batch+50])
#     }
#     response=requests.get(channel_url,params=params_channels).json()
#     print(len(response['items']))
#     break

In [120]:
channels_data = pd.DataFrame({"channel_id":[data['id'] for data in all_channel_data],
                 "channel_title":[data['snippet'].get('title') for data in all_channel_data],
                 "channel_description":[data['snippet'].get('description') for data in all_channel_data],
                 "channel_creation_date":[data['snippet'].get('publishedAt') for data in all_channel_data], 
                 "channel_country":[data['snippet'].get('country') for data in all_channel_data],
                 "thumbnail":[data['snippet']['thumbnails']['default']['url'] for data in all_channel_data],
                 "subscriberCount":[data['statistics'].get('subscriberCount') for data in all_channel_data],
                 "totalViewCount":[data['statistics'].get('viewCount') for data in all_channel_data],
                 "totalVideoCount":[data['statistics'].get('videoCount') for data in all_channel_data]}) 

In [122]:
channels_data.to_csv("channels.csv")